# ⚖️ Week 6 hands-on: RAG regression comparisons

## Learning Objectives
In this notebook, you will learn:
1. **A paired scorecard** - run baseline v1 and candidate v2 on the same cases and compare quality, safety, cost and latency
2. **Hidden regressions** - find the slice and the cases that got worse while the average improved
3. **The gold-context swap** - decide whether retrieval or generation caused each failure
4. **Ablation** - remove one component at a time and measure what it earns
5. **A rule-based release decision** - block v2, ship v2.1, and test the limits of the method

## Where this fits
This is the runnable companion to `week-06-rag-regression-comparisons-handson.md`, using the same code as `lab.py` (a test keeps the two identical). Read `week-06-rag-regression-comparisons-theory.md` first. It reuses Week 4's retrieval metrics and Week 5's answer checks unchanged, and closes Phase 2.

## Prerequisites
- `week-06-rag-regression-comparisons-theory.md`, and Weeks 4 and 5
- The course environment from `new_course/setup.md`, selected as this notebook's kernel
- No API keys. Cost and latency are simulated from illustrative constants

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, imports the pipeline versions, and loads Week 4's and Week 5's labs so their metrics and checks run unchanged.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Versions, corpus and the reused labs
# ============================================================================
import dataclasses
import importlib.util
import sys
from collections import defaultdict
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.rag_pipeline import VERSIONS, Pipeline  # noqa: E402
from shared.retrieval import Chunk, chunk_documents, load_documents  # noqa: E402


def load_week_lab(folder: str, name: str):
    """Week 6 reuses Week 4's metrics and Week 5's checks unchanged, so it loads their labs."""
    spec = importlib.util.spec_from_file_location(name, COURSE / "weeks" / folder / "handson" / "lab.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


W4 = load_week_lab("week-04-retrieval-evaluation", "week04_lab_for_w6")


W5 = load_week_lab("week-05-answer-quality-and-evidence", "week05_lab_for_w6")


CHUNKS = chunk_documents(load_documents())


BY_ID = {c.chunk_id: c for c in CHUNKS}


POLICY = {
    "max_critical_failures": 0,         # internal text in an answer
    "max_slice_answer_drop": 0,         # passing answer cases lost on any slice
    "max_slice_recall_drop": 0.05,      # Recall@3 lost on any retrieval slice
    "max_p95_latency_increase": 0.20,   # simulated p95 at most 20% above the baseline's
}


print(f"✅ Environment ready: versions {list(VERSIONS)}, {len(CHUNKS)} chunks")

---
## 📋 Part 1: A Paired Scorecard

Both versions run on Week 4's 50 queries and Week 5's 24 answer cases. The scorecard puts quality, safety, cost and latency side by side. Rows marked SIMULATED come from illustrative constants.

In [ ]:
# ============================================================================
# SCORECARD: v1 against v2 on the same cases
# ============================================================================
def retrieval_rows(version: str) -> list[dict]:
    pipeline = Pipeline(VERSIONS[version], CHUNKS)
    rows = []
    for q in W4.read_jsonl(W4.QUERIES):
        _searched, hits = pipeline.retrieve(q["query"])
        rows.append({"id": q["query_id"], "slice": q["slice"], **W4.score_query(hits, q["relevance"], 3)})
    return rows


def answer_rows(version: str) -> list[dict]:
    pipeline = Pipeline(VERSIONS[version], CHUNKS)
    rows = []
    for case in W5.read_jsonl(W5.CASES):
        run = pipeline.run(case["question"])
        results = [check(case, run.evidence, run.answer) for check in W5.CHECKS]
        rows.append({"id": case["case_id"], "slice": case["slice"], "case": case, "run": run,
                     "results": results, "passed": all(r.passed for r in results),
                     "critical": any(not r.passed and r.severity == "critical" for r in results)})
    return rows


def percentile(values: list[float], pct: float) -> float:
    """Nearest-rank percentile: the smallest value with at least pct% of values at or below it."""
    ordered = sorted(values)
    index = max(0, min(len(ordered) - 1, -(-len(ordered) * pct // 100) - 1))
    return ordered[int(index)]


def summarize(version: str) -> dict:
    ret, ans = retrieval_rows(version), answer_rows(version)
    latencies = [r["run"].latency_ms for r in ans]
    return {
        "version": version, "retrieval": ret, "answers": ans,
        "recall@3": W4.mean(ret, "recall"), "mrr": W4.mean(ret, "mrr"), "ndcg@3": W4.mean(ret, "ndcg"),
        "violations": sum(r["violations"] for r in ret),
        "answers_passed": sum(r["passed"] for r in ans), "critical": sum(r["critical"] for r in ans),
        "abstained": sum(r["run"].answer.abstained for r in ans),
        "cost_per_1k": 1000 * sum(r["run"].cost_usd for r in ans) / len(ans),
        "p50_ms": percentile(latencies, 50), "p95_ms": percentile(latencies, 95),
    }


def print_scorecard(base: dict, cand: dict) -> None:
    rows = [("Retrieval Recall@3", "recall@3", "{:.3f}"), ("Retrieval MRR", "mrr", "{:.3f}"),
            ("Retrieval nDCG@3", "ndcg@3", "{:.3f}"), ("Filter violations (150 slots)", "violations", "{:d}"),
            ("Answer cases passing all checks (24)", "answers_passed", "{:d}"),
            ("Critical failures", "critical", "{:d}"), ("Answers abstained", "abstained", "{:d}"),
            ("SIMULATED cost per 1,000 answers ($)", "cost_per_1k", "{:.3f}"),
            ("SIMULATED p50 latency (ms)", "p50_ms", "{:.0f}"), ("SIMULATED p95 latency (ms)", "p95_ms", "{:.0f}")]
    print(f"{'metric':<40}{base['version']:>10}{cand['version']:>10}")
    for label, key, fmt in rows:
        print(f"{label:<40}{fmt.format(base[key]):>10}{fmt.format(cand[key]):>10}")


V1, V2, V21 = summarize("v1"), summarize("v2"), summarize("v2.1")
print_scorecard(V1, V2)

Every quality and safety row improves. v2 also abstains 8 times, and that is why its simulated cost and p50 latency fall.

---
## 🔍 Part 2: Hidden Regressions

Slices and pairs show what the averages blend away.

> **Key Insight**: v2 gains 7 answer cases overall and loses one on the normal slice, the everyday path.

In [ ]:
# ============================================================================
# SLICES AND PAIRS: Where v2 got worse
# ============================================================================
def slice_deltas(base: dict, cand: dict) -> list[dict]:
    """Per slice: answer cases passed and retrieval Recall@3, for both versions."""
    out = []
    for kind, rows_key, metric in (("answers", "answers", "passed"), ("retrieval", "retrieval", "recall")):
        groups_b, groups_c = defaultdict(list), defaultdict(list)
        for r in base[rows_key]:
            groups_b[r["slice"]].append(r)
        for r in cand[rows_key]:
            groups_c[r["slice"]].append(r)
        for name in groups_b:
            b = sum(r[metric] for r in groups_b[name])
            c = sum(r[metric] for r in groups_c[name])
            if kind == "retrieval":
                b, c = b / len(groups_b[name]), c / len(groups_c[name])
            out.append({"kind": kind, "slice": name, "n": len(groups_b[name]), "base": b, "cand": c, "delta": c - b})
    return out


def paired(base_rows: list[dict], cand_rows: list[dict], metric: str) -> dict[str, list[str]]:
    """Case by case on the same IDs: which improved, which regressed."""
    cand = {r["id"]: r for r in cand_rows}
    better = [r["id"] for r in base_rows if cand[r["id"]][metric] > r[metric]]
    worse = [r["id"] for r in base_rows if cand[r["id"]][metric] < r[metric]]
    return {"improved": better, "regressed": worse}


def print_slices(base: dict, cand: dict) -> None:
    print(f"{'kind':<10}{'slice':<26}{'n':>3}{base['version']:>9}{cand['version']:>9}{'delta':>9}")
    for d in slice_deltas(base, cand):
        fmt = "{:>9.3f}" if d["kind"] == "retrieval" else "{:>9.0f}"
        flag = "  <- regression" if d["delta"] < 0 else ""
        print(f"{d['kind']:<10}{d['slice']:<26}{d['n']:>3}" + fmt.format(d["base"]) + fmt.format(d["cand"])
              + ("{:>+9.3f}" if d["kind"] == "retrieval" else "{:>+9.0f}").format(d["delta"]) + flag)


def print_paired(base: dict, cand: dict) -> None:
    ans = paired(base["answers"], cand["answers"], "passed")
    ret = paired(base["retrieval"], cand["retrieval"], "ndcg")
    print(f"Answer cases ({base['version']} -> {cand['version']}): {len(ans['improved'])} improved, "
          f"{len(ans['regressed'])} regressed")
    print(f"  regressed: {ans['regressed']}")
    print(f"Retrieval queries, nDCG@3: {len(ret['improved'])} improved, {len(ret['regressed'])} regressed")
    print(f"  regressed: {ret['regressed']}")


print_slices(V1, V2)
print()
print_paired(V1, V2)

---
## 🔧 Part 3: The Gold-Context Swap

Rerun each failing case with only its gold sections as evidence. Passing means retrieval caused the failure. Failing anyway means generation did.

In [ ]:
# ============================================================================
# ATTRIBUTION: Retrieval or generation
# ============================================================================
def attribute(version: str) -> list[dict]:
    """Gold-context swap: rerun each failing answer case with only its gold sections as evidence.

    Passing with gold evidence means retrieval caused the failure. Failing anyway means generation did.
    """
    pipeline = Pipeline(VERSIONS[version], CHUNKS)
    out = []
    for row in answer_rows(version):
        if row["passed"]:
            continue
        case = row["case"]
        gold = [BY_ID[s] for s in case["gold_sections"]]
        swapped = pipeline.run(case["question"], evidence=gold)
        results = [check(case, gold, swapped.answer) for check in W5.CHECKS]
        cause = "retrieval" if all(r.passed for r in results) else "generation"
        out.append({"id": row["id"], "slice": row["slice"], "cause": cause,
                    "failed": [r.check for r in row["results"] if not r.passed],
                    "failed_with_gold": [r.check for r in results if not r.passed]})
    return out


def print_attribution(version: str) -> None:
    print(f"Gold-context swap for {version}'s failing answer cases:")
    for r in attribute(version):
        print(f"  {r['id']} ({r['slice']}): {r['cause']:<10} failed {r['failed']}, with gold evidence {r['failed_with_gold']}")


print_attribution("v1")
print()
print_attribution("v2")

v1 fails mostly in search. Every v2 failure is generation: the 3.0 abstention threshold rejects even the right evidence.

---
## 🧮 Part 4: Ablation

Take v2.1 (threshold 2.0) apart one component at a time.

In [ ]:
# ============================================================================
# ABLATION: What each component earns
# ============================================================================
def ablate(version: str) -> list[dict]:
    """Turn off one component of a version at a time and rerun both evaluations."""
    config = VERSIONS[version]
    variants = [("as configured", config)]
    for field, off in (("rewrite", False), ("rerank", False), ("filtered", False)):
        if getattr(config, field) != off:
            variants.append((f"without {field}", dataclasses.replace(config, name=f"{version}-no-{field}", **{field: off})))
    out = []
    for label, variant in variants:
        VERSIONS[variant.name] = variant
        summary = summarize(variant.name)
        out.append({"label": label, **{k: summary[k] for k in
                    ("ndcg@3", "recall@3", "answers_passed", "critical", "p50_ms")}})
        if variant is not config:
            del VERSIONS[variant.name]
    return out


def print_ablation(version: str) -> None:
    print(f"Ablation of {version}: one component off at a time")
    print(f"{'variant':<20}{'nDCG@3':>8}{'R@3':>8}{'answers':>9}{'critical':>9}{'p50 ms':>8}")
    for r in ablate(version):
        print(f"{r['label']:<20}{r['ndcg@3']:>8.3f}{r['recall@3']:>8.3f}{r['answers_passed']:>6}/24"
              f"{r['critical']:>9}{r['p50_ms']:>8.0f}")
    print("Latency is simulated.")


print_ablation("v2.1")

Without the reranker, retrieval metrics rise and answers fall from 22 to 19. Without rewriting, nothing gets worse. The outcome decides, not the proxy.

---
## ✅ Part 5: A Rule-Based Release Decision

`POLICY` was written before the comparison. Any failed rule blocks the candidate.

In [ ]:
# ============================================================================
# DECIDE: Block v2, ship v2.1
# ============================================================================
def decide(base: dict, cand: dict) -> tuple[str, list[str]]:
    """Apply POLICY. Any failed rule blocks the candidate."""
    reasons = []
    if cand["critical"] > POLICY["max_critical_failures"]:
        reasons.append(f"{cand['critical']} critical failures (allowed {POLICY['max_critical_failures']})")
    for d in slice_deltas(base, cand):
        if d["kind"] == "answers" and d["delta"] < -POLICY["max_slice_answer_drop"]:
            reasons.append(f"answer slice '{d['slice']}' drops from {d['base']} to {d['cand']} of {d['n']}")
        if d["kind"] == "retrieval" and d["delta"] < -POLICY["max_slice_recall_drop"]:
            reasons.append(f"retrieval slice '{d['slice']}' Recall@3 drops by {-d['delta']:.3f}")
    budget = base["p95_ms"] * (1 + POLICY["max_p95_latency_increase"])
    if cand["p95_ms"] > budget:
        reasons.append(f"simulated p95 latency {cand['p95_ms']:.0f} ms (budget {budget:.0f} ms)")
    return ("BLOCK" if reasons else "SHIP"), reasons


def print_decision(base: dict, cand: dict) -> None:
    verdict, reasons = decide(base, cand)
    print(f"Release decision for {cand['version']} against {base['version']}: {verdict}")
    for reason in reasons:
        print(f"  - {reason}")
    print("Thresholds are illustrative (POLICY in lab.py). Latency and cost are simulated.")


print_decision(V1, V2)
print()
print_decision(V1, V21)

### 5.1 ✍️ Your threshold
Exercise 2: try `min_score=2.5`. Does it ship? Is it the version you would ship?

In [ ]:
# ============================================================================
# YOUR VERSION: A different abstention threshold
# ============================================================================
VERSIONS["v2.5t"] = dataclasses.replace(VERSIONS["v2.1"], name="v2.5t", min_score=2.5)
mine = summarize("v2.5t")
print_decision(V1, mine)
print(f"🧮 answers passing: v2.1 {V21['answers_passed']}, v2.5t {mine['answers_passed']}")
del VERSIONS["v2.5t"]

### 5.2 🐛 Drill 1: the candidate was run on fewer cases
Score v2 on the first 20 cases only. Its pass rate looks far better, and pairing by case ID refuses the comparison.

In [ ]:
# ============================================================================
# DRILL 1: Unpaired case sets
# ============================================================================
subset = answer_rows("v2")[:20]
print(f"⚠️ v2 on 20 cases: {sum(r['passed'] for r in subset)}/20 against v1 on 24: {V1['answers_passed']}/24")
try:
    paired(V1["answers"], subset, "passed")
except KeyError as missing:
    print(f"❌ paired() refuses: no candidate result for {missing}")

### 5.3 🐛 Drill 2: the swap blames retrieval for an abstain case
A case with no gold sections gets empty evidence in the swap, and an abstaining writer always passes it. Report those cases as inconclusive.

In [ ]:
# ============================================================================
# DRILL 2: Mark empty-gold swaps as inconclusive
# ============================================================================
def attribute_with_limits(version):
    rows = attribute(version)
    cases = {c["case_id"]: c for c in W5.read_jsonl(W5.CASES)}
    for r in rows:
        if not cases[r["id"]]["gold_sections"]:
            r["cause"] = "inconclusive"
    return rows


for r in attribute_with_limits("v2.1"):
    print(f"🔧 {r['id']} ({r['slice']}): {r['cause']}")

### 5.4 ✍️ Measure drift
Exercise 4: add a 2026-04 electronics refund rule to the corpus and rerun the v2.1 retrieval. The cell restores the corpus afterwards.

In [ ]:
# ============================================================================
# DRIFT: A new policy chunk changes the rankings
# ============================================================================
before = {r["id"]: r["ndcg"] for r in retrieval_rows("v2.1")}
CHUNKS.append(Chunk("refund-policy-2026-04#R2", "refund-policy-2026-04", "Refund policy", "current", "public",
                    ("refund-policy-2026-04#R2",),
                    "Refund window. A customer can request a refund within 14 days of delivery for electronics. "
                    "Day 14 is the last eligible day for electronics."))
after = {r["id"]: r["ndcg"] for r in retrieval_rows("v2.1")}
CHUNKS.pop()
changed = {q: (round(before[q], 2), round(after[q], 2)) for q in before if abs(before[q] - after[q]) > 1e-9}
print(f"🔍 nDCG@3 changed on {len(changed)} queries: {changed}")

---
## 📝 Summary

In this notebook, we learned:

### 1. A paired scorecard
- Same cases, pinned versions, and safety, cost and latency beside quality. v2 looks cheaper only because it abstains.

### 2. Hidden regressions
- The normal slice drops from 7 to 6, and four retrieval queries get worse, while every average rises.

### 3. The gold-context swap
- v1 fails mostly in search, and v2 entirely in the writer. Empty-gold cases are inconclusive.

### 4. Ablation
- The reranker trades retrieval metrics for better answers. Rewriting earns nothing on this set.

### 5. A rule-based release decision
- v2 is blocked by one slice rule. v2.1 ships inside its latency budget.

### Next Steps
- **`week-07-tools-and-side-effects-handson.ipynb`**: checks tool calls and the refunds they actually store.